# Assignment 3 (Q1 + Q2) — Telco Customer Churn: Classification and PCA / SVD
**Data Science Lab (CS2311)**

- **Q1:** Predict whether a customer will churn (leave the company). Preprocess, explore, train 6 classifiers, and evaluate each with a confusion matrix, Accuracy, Precision, Recall, F1, ROC-AUC, ROC curves and Precision–Recall curves.
- **Q2:** Implement PCA and SVD (from scratch with NumPy, checked against scikit-learn), reduce the features, repeat the same experiments, and compare original vs PCA vs SVD on performance, number of features and time.

Q2 reuses everything from Q1, so both live in one notebook.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## Loading the data

The Kaggle "Telco Customer Churn" dataset (`WA_Fn-UseC_-Telco-Customer-Churn.csv`) is IBM sample data. The same file is published on IBM's GitHub, so pandas can read it straight from the URL (needs internet). If you downloaded it from Kaggle, put the CSV next to this notebook and it will be used instead.

In [ ]:
import os

LOCAL_FILE = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

df_raw = pd.read_csv(LOCAL_FILE if os.path.exists(LOCAL_FILE) else URL)
print('Shape:', df_raw.shape)
df_raw.head()

**Attributes:** demographics (`gender`, `SeniorCitizen`, `Partner`, `Dependents`), account info (`tenure` in months, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`), services (`PhoneService`, `InternetService`, `OnlineSecurity`, `TechSupport`, streaming, ...), and the target **`Churn`** (Yes/No).

---
# Question 1 — Classification
## 1.1 Preprocessing

**Known traps in this dataset:**
1. `customerID` is a unique ID with no predictive value → drop it.
2. `TotalCharges` is stored as **text**, and 11 rows contain a blank `' '` (brand-new customers with tenure 0). `pd.to_numeric(..., errors='coerce')` turns blanks into NaN, which we can then handle.

In [ ]:
df_raw.info()

In [ ]:
df = df_raw.copy()
df = df.drop(columns='customerID')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing values after converting TotalCharges:\n', df.isnull().sum()[df.isnull().sum() > 0], sep='')
print('\nTenure of the rows with missing TotalCharges:', df.loc[df.TotalCharges.isnull(), 'tenure'].unique())

The missing `TotalCharges` all belong to customers with **tenure = 0**: they have not been billed yet, so the logical value is **0** (better than the median, which would invent charges they never paid).

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

dups = df.duplicated().sum()
print('Duplicate rows:', dups)
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

**Duplicates:** after dropping `customerID`, a few rows are identical. With 19 mostly yes/no columns, two different customers can genuinely have the same profile, so removing them is optional; we remove them to follow the assignment, and the effect on 7,000 rows is negligible.

## 1.2 Exploratory data analysis

In [ ]:
churn_counts = df['Churn'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x='Churn', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Target distribution (count)')
ax[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'])
ax[1].set_title('Target distribution (share)')
plt.tight_layout()
plt.show()
print(churn_counts)

**The classes are imbalanced:** about 26% churn and 74% stay. A model that always predicts "No" would score about 74% accuracy while catching zero churners. That is why we judge models mainly by **Recall, F1 and ROC-AUC**, not accuracy alone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=ax, element='step', stat='density', common_norm=False)
    ax.set_title(f'{col} by churn')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols_to_plot = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity', 'SeniorCitizen']
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.ravel(), cat_cols_to_plot):
    rate = df.groupby(col)['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).sort_values()
    rate.plot(kind='barh', ax=ax, color='tab:orange')
    ax.set_title(f'Churn rate (%) by {col}')
    ax.set_xlabel('% churned')
plt.tight_layout()
plt.show()

**EDA insights:**
1. Churners have **low tenure**: most leave in the first months.
2. Churners pay **higher monthly charges**.
3. **Month-to-month** contracts churn far more than one- or two-year contracts.
4. **Fiber optic** customers and **electronic check** payers churn the most.
5. Customers **without tech support or online security** churn more; senior citizens churn more.

## 1.3 Encoding and scaling

- **Target:** `Churn` Yes/No → 1/0.
- **Categorical features:** **one-hot encoding** with `pd.get_dummies(drop_first=True)`. Each category becomes a 0/1 column; `drop_first` removes one redundant column per feature (if a customer is not "One year" and not "Two year", they must be "Month-to-month").
- **Train/test split (80/20)** with `stratify=y`, so both sets keep the same 26% churn ratio.
- **StandardScaler:** fit on the training set only, then transform both (avoids data leakage). KNN, SVM and Logistic Regression need it because they use distances or gradients.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)
print('Columns after one-hot encoding:', df_encoded.shape[1] - 1, 'features + 1 target')

X = df_encoded.drop(columns='Churn')
y = df_encoded['Churn']
feature_names = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print('Train:', X_train_s.shape, ' Test:', X_test_s.shape)
print('Churn rate — train: {:.3f}, test: {:.3f}'.format(y_train.mean(), y_test.mean()))

## 1.4 The confusion matrix and the metrics

"Positive" = churn (the thing we want to catch).

|  | Predicted No | Predicted Yes |
|---|---|---|
| **Actually No** | TN | FP (false alarm) |
| **Actually Yes** | FN (missed churner) | TP |

| Metric | Formula | Question it answers |
|---|---|---|
| **Accuracy** | (TP + TN) / total | How often is the model right overall? |
| **Precision** | TP / (TP + FP) | When it says "churn", how often is it right? |
| **Recall** | TP / (TP + FN) | Of all real churners, how many did it catch? |
| **F1** | 2·P·R / (P + R) | Balance between precision and recall |
| **ROC-AUC** | area under the ROC curve | Probability that a random churner gets a higher churn score than a random non-churner (0.5 = coin flip, 1 = perfect) |

**ROC curve:** True Positive Rate (recall) vs False Positive Rate across **all** thresholds. **Precision–Recall curve:** precision vs recall across all thresholds; more informative than ROC when the positive class is rare. Both need predicted **probabilities** (`predict_proba(X)[:, 1]` = probability of class 1), not just 0/1 labels.

In [ ]:
def get_models():
    """Fresh, untrained copies of the 6 classifiers."""
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'KNN (k=15)':          KNeighborsClassifier(n_neighbors=15),
        'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
        'Naive Bayes':         GaussianNB(),
        'SVM (RBF)':           SVC(probability=True, random_state=RANDOM_STATE),  # probability=True enables predict_proba
    }


def run_experiment(X_tr, X_te, y_tr, y_te, tag):
    """Train every model, plot confusion matrices + ROC + PR curves, return a metrics table."""
    models = get_models()
    rows, curves = [], {}
    fig_cm, axes_cm = plt.subplots(2, 3, figsize=(15, 9))

    for ax, (name, model) in zip(axes_cm.ravel(), models.items()):
        start = time.perf_counter()
        model.fit(X_tr, y_tr)
        train_time = time.perf_counter() - start

        y_pred = model.predict(X_te)                 # 0/1 labels (threshold 0.5)
        y_proba = model.predict_proba(X_te)[:, 1]    # probability of churn

        cm = confusion_matrix(y_te, y_pred)
        ConfusionMatrixDisplay(cm, display_labels=['No churn', 'Churn']).plot(ax=ax, cmap='Blues', colorbar=False)
        ax.set_title(name)

        rows.append({
            'Model': name,
            'Accuracy': accuracy_score(y_te, y_pred),
            'Precision': precision_score(y_te, y_pred),
            'Recall': recall_score(y_te, y_pred),
            'F1': f1_score(y_te, y_pred),
            'ROC-AUC': roc_auc_score(y_te, y_proba),
            'Avg precision (PR-AUC)': average_precision_score(y_te, y_proba),
            'Train time (s)': train_time,
            'TN, FP, FN, TP': tuple(int(v) for v in cm.ravel()),
        })
        curves[name] = y_proba

    fig_cm.suptitle(f'Confusion matrices — {tag}', fontsize=15)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    for name, proba in curves.items():
        fpr, tpr, _ = roc_curve(y_te, proba)
        ax[0].plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_te, proba):.3f})')
        prec, rec, _ = precision_recall_curve(y_te, proba)
        ax[1].plot(rec, prec, label=f'{name} (AP = {average_precision_score(y_te, proba):.3f})')
    ax[0].plot([0, 1], [0, 1], 'k--', label='Random guess')
    ax[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate (Recall)', title=f'ROC curves — {tag}')
    ax[1].axhline(y_te.mean(), color='k', ls='--', label=f'Random guess ({y_te.mean():.2f})')
    ax[1].set(xlabel='Recall', ylabel='Precision', title=f'Precision–Recall curves — {tag}')
    ax[0].legend(fontsize=9); ax[1].legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    return pd.DataFrame(rows).set_index('Model').sort_values('ROC-AUC', ascending=False)

**Why a function?** Q2 asks us to repeat exactly the same experiment on PCA- and SVD-reduced features. Writing it once as `run_experiment()` means Q2 is just two more calls.

**The six classifiers in one line each:**
- **Logistic Regression:** a weighted sum of features passed through a sigmoid to give a probability between 0 and 1.
- **KNN:** looks at the k most similar customers in the training set and takes a vote.
- **Decision Tree:** a flowchart of yes/no questions ("Contract = month-to-month?"). `max_depth` stops it memorising the data.
- **Random Forest:** many trees trained on random subsets, then a majority vote.
- **Naive Bayes:** Bayes' rule, assuming features are independent given the class. Very fast.
- **SVM:** finds the boundary with the widest margin between classes; the RBF kernel allows curved boundaries.

In [ ]:
results_original = run_experiment(X_train_s, X_test_s, y_train, y_test, 'Original features')
results_original.round(4)

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
results_original[metric_cols].plot(kind='bar', figsize=(14, 5), rot=15)
plt.title('Comparison of all evaluation metrics (original features)')
plt.ylim(0, 1)
plt.legend(loc='lower right', ncol=5)
plt.show()

best_model = results_original['ROC-AUC'].idxmax()
print('Best model by ROC-AUC:', best_model)
print(results_original.loc[best_model, metric_cols].round(4))

## Q1 Conclusion

- **Logistic Regression** gives the best ROC-AUC (about 0.84), with Random Forest close behind. The table above shows the exact numbers; SVM, despite good accuracy, ranks churners less well (lower ROC-AUC).
- **Accuracy is misleading here:** most models reach 78–80% accuracy, only a few points above the 74% you would get by always predicting "No churn".
- **Naive Bayes** has by far the highest recall (it catches the most churners) but low precision and only about 65% accuracy (many false alarms). It assumes the features are independent, which is false here (e.g. tenure and TotalCharges are strongly linked).
- **Business view:** missing a churner (FN) usually costs more than a false alarm (FP, an unnecessary discount offer), so a model with good ROC-AUC whose threshold is tuned for higher recall is the most useful choice.

---
# Question 2 — Dimensionality reduction with PCA and SVD

## Intuition

**Analogy:** photographing a 3-D object for an ID card. You rotate it until the photo shows the most detail (the widest spread). That best direction is **principal component 1**. The next-best direction at right angles to it is PC2, and so on. Directions where the object is almost flat carry little information, so we drop them.

After one-hot encoding we have 30 features, and many are redundant (for example, "No internet service" appears in six different columns and they always agree). PCA merges correlated columns into fewer, uncorrelated **components**.

**Explained variance ratio:** the fraction of the total variance each component captures. We keep enough components to reach about **95%** cumulative variance.

## The algorithms

**PCA (eigen-decomposition):**
1. Centre the data: Xc = X − mean (our data is already standardised).
2. Covariance matrix: C = Xcᵀ·Xc / (n − 1), a d × d matrix.
3. Eigenvalues and eigenvectors of C.
4. Sort by eigenvalue, largest first (eigenvalue = variance along that direction).
5. Keep the top k eigenvectors W (d × k) and project: Z = Xc·W (n × k).

**SVD:** factor the centred data matrix directly: Xc = U·S·Vᵀ. The rows of Vᵀ are the same principal directions, and S²/(n − 1) are the same eigenvalues. Projection: Z = Xc·Vᵀ[:k]ᵀ.

**Difference:** PCA builds the covariance matrix and eigen-decomposes it; SVD works on the data matrix directly (numerically more stable; scikit-learn's PCA uses SVD internally). scikit-learn's `TruncatedSVD` does **not** centre the data, which is why it also works on sparse data.

In [ ]:
class PCAScratch:
    """PCA via eigen-decomposition of the covariance matrix."""

    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_                               # 1. centre
        cov = np.cov(Xc, rowvar=False)                    # 2. covariance (d x d)
        eigvals, eigvecs = np.linalg.eigh(cov)            # 3. eigh: for symmetric matrices
        order = np.argsort(eigvals)[::-1]                 # 4. sort, largest first
        eigvals, eigvecs = eigvals[order], eigvecs[:, order]
        self.explained_variance_ = eigvals
        self.explained_variance_ratio_ = eigvals / eigvals.sum()
        self.components_ = eigvecs[:, :self.n_components].T   # k x d
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) @ self.components_.T   # 5. project

    def fit_transform(self, X):
        return self.fit(X).transform(X)


class SVDScratch:
    """Dimensionality reduction via Singular Value Decomposition of the centred data."""

    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)    # Xc = U · diag(S) · Vt
        self.singular_values_ = S
        self.explained_variance_ratio_ = S**2 / np.sum(S**2)
        self.components_ = Vt[:self.n_components]            # k x d
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

### Check: our implementations agree with scikit-learn
Components can come out with **flipped signs** (a direction and its opposite are the same axis), so we compare absolute values.

In [ ]:
k_check = 5
pca_mine = PCAScratch(k_check).fit(X_train_s)
svd_mine = SVDScratch(k_check).fit(X_train_s)
pca_sk = PCA(k_check).fit(X_train_s)

print('Explained variance ratio (first 5):')
print('  PCA scratch :', np.round(pca_mine.explained_variance_ratio_[:5], 5))
print('  SVD scratch :', np.round(svd_mine.explained_variance_ratio_[:5], 5))
print('  sklearn PCA :', np.round(pca_sk.explained_variance_ratio_, 5))
print('Projections match sklearn (up to sign):',
      np.allclose(np.abs(pca_mine.transform(X_test_s)), np.abs(pca_sk.transform(X_test_s)), atol=1e-6),
      np.allclose(np.abs(svd_mine.transform(X_test_s)), np.abs(pca_sk.transform(X_test_s)), atol=1e-6))

## Choosing the number of components
Scree plot (variance per component) and cumulative variance; we pick the smallest k that keeps ≥ 95% of the variance.

In [ ]:
full = PCAScratch(X_train_s.shape[1]).fit(X_train_s)
ratio = full.explained_variance_ratio_
cumulative = np.cumsum(ratio)
k = int(np.argmax(cumulative >= 0.95) + 1)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].bar(range(1, len(ratio) + 1), ratio, color='tab:blue')
ax[0].set(xlabel='Component', ylabel='Explained variance ratio', title='Scree plot')
ax[1].plot(range(1, len(cumulative) + 1), cumulative, marker='o')
ax[1].axhline(0.95, color='r', ls='--', label='95% threshold')
ax[1].axvline(k, color='g', ls='--', label=f'k = {k}')
ax[1].set(xlabel='Number of components', ylabel='Cumulative explained variance', title='Cumulative variance')
ax[1].legend()
plt.tight_layout()
plt.show()

print(f'Original features: {X_train_s.shape[1]}  →  components kept: {k} '
      f'({cumulative[k-1]*100:.1f}% of the variance)')
print('Number of near-zero eigenvalues (perfectly redundant directions):', int((full.explained_variance_ < 1e-10).sum()))

The last few eigenvalues are essentially **zero**: those directions carry no information at all. They come from the "No internet service" columns, which are exact copies of each other. PCA detects this redundancy automatically.

In [ ]:
Z2 = PCAScratch(2).fit_transform(X_train_s)
plt.figure(figsize=(8, 6))
plt.scatter(Z2[y_train.values == 0, 0], Z2[y_train.values == 0, 1], s=5, alpha=0.4, label='No churn')
plt.scatter(Z2[y_train.values == 1, 0], Z2[y_train.values == 1, 1], s=5, alpha=0.4, label='Churn')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Training data projected onto the first 2 principal components')
plt.legend()
plt.show()

## Transform the data

Same rule as scaling: **fit on the training set only**, then transform both sets. We use our own implementations; `sklearn.decomposition.PCA` and `TruncatedSVD` would give the same results (shown above).

In [ ]:
start = time.perf_counter()
pca = PCAScratch(k).fit(X_train_s)
X_train_pca, X_test_pca = pca.transform(X_train_s), pca.transform(X_test_s)
pca_time = time.perf_counter() - start

start = time.perf_counter()
svd = SVDScratch(k).fit(X_train_s)
X_train_svd, X_test_svd = svd.transform(X_train_s), svd.transform(X_test_s)
svd_time = time.perf_counter() - start

print(f'PCA: {X_train_pca.shape}, fit+transform time = {pca_time*1000:.1f} ms')
print(f'SVD: {X_train_svd.shape}, fit+transform time = {svd_time*1000:.1f} ms')

## Repeat the classification experiments
### PCA-reduced features

In [ ]:
results_pca = run_experiment(X_train_pca, X_test_pca, y_train, y_test, f'PCA ({k} components)')
results_pca.round(4)

### SVD-reduced features

In [ ]:
results_svd = run_experiment(X_train_svd, X_test_svd, y_train, y_test, f'SVD ({k} components)')
results_svd.round(4)

## Comparison: original vs PCA vs SVD

In [ ]:
all_results = pd.concat({
    f'Original ({X_train_s.shape[1]} features)': results_original,
    f'PCA ({k} components)': results_pca,
    f'SVD ({k} components)': results_svd,
}, names=['Feature set', 'Model']).reset_index()

for metric in ['ROC-AUC', 'F1', 'Recall', 'Accuracy', 'Train time (s)']:
    print(f'\n=== {metric} ===')
    print(all_results.pivot(index='Model', columns='Feature set', values=metric).round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, metric in zip(axes.ravel(), ['ROC-AUC', 'F1', 'Recall', 'Train time (s)']):
    all_results.pivot(index='Model', columns='Feature set', values=metric).plot(kind='bar', ax=ax, rot=20)
    ax.set_title(metric)
    if metric != 'Train time (s)':
        ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
summary = all_results.groupby('Feature set').agg(
    avg_ROC_AUC=('ROC-AUC', 'mean'), best_ROC_AUC=('ROC-AUC', 'max'),
    avg_F1=('F1', 'mean'), total_train_time_s=('Train time (s)', 'sum'))
summary.insert(0, '#Features', [X_train_s.shape[1] if f.startswith('Original') else k for f in summary.index])
summary.round(4)

## Q2 Conclusion

- **PCA and SVD give practically identical results.** Our data is already centred (standardised), and on centred data SVD and PCA find exactly the same directions.
- **Fewer features:** the 30 original features are reduced to the number of components printed above while keeping 95% of the variance. Several original directions had zero variance (exact duplicate columns).
- **Performance is maintained:** with 17 instead of 30 features, ROC-AUC drops by only about 0.003–0.01 for Logistic Regression, KNN, SVM and Random Forest. Dimensionality reduction did **not** improve the best model, but it did not meaningfully hurt it either.
- **Tree models lose the most:** the decision tree drops by about 0.03 ROC-AUC. Trees split on single, meaningful columns ("Contract = month-to-month"); PCA components are blends of many columns, which makes these clean splits harder.
- **Naive Bayes gets *better*:** its accuracy and F1 rise after PCA. Naive Bayes assumes features are independent, and principal components are uncorrelated by construction, so its assumption fits the transformed data much better.
- **Time:** total training time drops only slightly here, because this dataset is small (the time table shows the numbers for your machine). The savings grow with bigger datasets and many more features.
- **Trade-off:** we lose **interpretability**, because a component is a mixture of many original features. Use PCA/SVD when speed, memory or multicollinearity matter, not when you need to explain which features drive churn.